In [2]:
!pip install pandas numpy matplotlib


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    r"C:\Users\SANKET THAKUR\PycharmProjects\PythonProject\Column_Chromatography_AI_ML\Data\Raw\rf_dataset_0203_total.csv"
)

# 1. Dataset dimensions
print("\nDATASET SHAPE")
print(df.shape)

# 2. Column names
print("\nCOLUMN NAMES")
for i, col in enumerate(df.columns):
    print(i, col)

# 3. Data types
print("\nDATA TYPES")
print(df.dtypes.value_counts())

# 4. Missing values
print("\nMISSING VALUES")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# 5. Duplicate rows
print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

# 6. First five observations
print("\nSAMPLE DATA")
print(df.head())

# 7. Numerical statistics
print("\nNUMERICAL SUMMARY")
print(df.describe().T)

# 8. Categorical columns
print("\nCATEGORICAL COLUMNS")
categorical = df.select_dtypes(include=["object", "category"]).columns

for col in categorical:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts().head(10))

# 9. Possible target columns
print("\nPOSSIBLE TARGET COLUMNS")
for col in df.columns:
    if any(term in col.lower() for term in
           ["rf", "solvent", "ratio", "composition", "fraction"]):
        print(col)


DATASET SHAPE
(22902, 10)

COLUMN NAMES
0 SMILES
1 Solvent_1
2 Solvent_2
3 Solvent_3
4 Fraction_1
5 Fraction_2
6 Fraction_3
7 Rf_value
8 doi
9 label

DATA TYPES
str        6
float64    4
Name: count, dtype: int64

MISSING VALUES
Solvent_3    22604
Solvent_2     2248
dtype: int64

DUPLICATE ROWS
1717

SAMPLE DATA
                                              SMILES      Solvent_1  \
0                     C(CCCCCCCCCCCCCCC)(=O)OCC(CO)O      Isohexane   
1              N(=[N+]=[N-])CCCCCCCCC/C=C/CCCCC(=O)O  Ethyl acetate   
2                 N(=[N+]=[N-])CCCCCCCCCCCCCCCC(=O)O  Ethyl acetate   
3  N(=[N+]=[N-])CCCCCCCCCC=CCCCCC(=O)OCC(COC(CCCC...  Ethyl acetate   
4  N(=[N+]=[N-])CCCCCCCCCCCCCCCC(=O)OCC(COC(CCCCC...  Ethyl acetate   

     Solvent_2 Solvent_3  Fraction_1  Fraction_2  Fraction_3  Rf_value  \
0      Acetone       NaN        0.75        0.25         0.0      0.30   
1  Cyclohexane       NaN        0.50        0.50         0.0      0.10   
2  Cyclohexane       NaN        0.50

C:\Users\SANKET THAKUR\AppData\Local\Temp\ipykernel_16748\2320596664.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include=["object", "category"]).columns


In [4]:

import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD DATASET
# ==========================================

DATA_PATH = (
    r"C:\Users\SANKET THAKUR\PycharmProjects"
    r"\PythonProject\Column_Chromatography_AI_ML"
    r"\Data\Raw\rf_dataset_0203_total.csv"
)

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)


# ==========================================
# 2. CHECK REQUIRED COLUMNS
# ==========================================

required_columns = [
    "SMILES",
    "Solvent_1", "Solvent_2", "Solvent_3",
    "Fraction_1", "Fraction_2", "Fraction_3",
    "Rf_value"
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")


# ==========================================
# 3. DATA TYPES
# ==========================================

fraction_columns = [
    "Fraction_1",
    "Fraction_2",
    "Fraction_3"
]

df[fraction_columns + ["Rf_value"]] = df[
    fraction_columns + ["Rf_value"]
].apply(pd.to_numeric, errors="coerce")


# ==========================================
# 4. MISSING VALUE ANALYSIS
# ==========================================

print("\nMissing values:")
print(df.isnull().sum())


# ==========================================
# 5. DUPLICATE ANALYSIS
# ==========================================

print("\nDuplicate records:", df.duplicated().sum())


# ==========================================
# 6. RF VALUE VALIDATION
# ==========================================

print("\nRf statistics:")
print(df["Rf_value"].describe())

invalid_rf = df[
    (df["Rf_value"] < 0) |
    (df["Rf_value"] > 1)
]

print("Invalid Rf records:", len(invalid_rf))


# ==========================================
# 7. SOLVENT FRACTION VALIDATION
# ==========================================

df["Fraction_Sum"] = df[fraction_columns].sum(
    axis=1,
    min_count=1
)

invalid_fraction = df[
    (df[fraction_columns].lt(0).any(axis=1)) |
    (df[fraction_columns].gt(1).any(axis=1)) |
    (~np.isclose(df["Fraction_Sum"], 1.0, atol=0.001))
]

print("\nInvalid fraction records:", len(invalid_fraction))


# ==========================================
# 8. SOLVENT SYSTEM CLASSIFICATION
# ==========================================

solvent_columns = [
    "Solvent_1",
    "Solvent_2",
    "Solvent_3"
]

df["Number_of_Solvents"] = df[
    solvent_columns
].notna().sum(axis=1)

print("\nSolvent system distribution:")
print(df["Number_of_Solvents"].value_counts().sort_index())


# ==========================================
# 9. MOLECULAR STRUCTURE VALIDATION
# ==========================================

print("\nMissing SMILES:", df["SMILES"].isnull().sum())

print("Unique molecules:", df["SMILES"].nunique())


# ==========================================
# 10. FINAL AUDIT
# ==========================================

print("\nDataset audit completed.")

print("Final shape:", df.shape)

print("\nSample records:")
print(
    df[
        required_columns +
        ["Fraction_Sum", "Number_of_Solvents"]
    ].head(10)
)


Original dataset shape: (22902, 10)

Missing values:
SMILES            0
Solvent_1         0
Solvent_2      2248
Solvent_3     22604
Fraction_1        0
Fraction_2        0
Fraction_3        0
Rf_value          0
doi               0
label             0
dtype: int64

Duplicate records: 1717

Rf statistics:
count    22902.000000
mean         0.453428
std          0.236128
min         -0.013000
25%          0.300000
50%          0.400000
75%          0.598000
max          1.007000
Name: Rf_value, dtype: float64
Invalid Rf records: 5

Invalid fraction records: 0

Solvent system distribution:
Number_of_Solvents
1     2248
2    20356
3      298
Name: count, dtype: int64

Missing SMILES: 0
Unique molecules: 15735

Dataset audit completed.
Final shape: (22902, 12)

Sample records:
                                              SMILES      Solvent_1  \
0                     C(CCCCCCCCCCCCCCC)(=O)OCC(CO)O      Isohexane   
1              N(=[N+]=[N-])CCCCCCCCC/C=C/CCCCC(=O)O  Ethyl acetate   
2  